<a href="https://colab.research.google.com/github/SamuelBG-SVG/Proyecto---Grietas/blob/main/02_Entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 02_entrenamiento_baseline.ipynb
# Proyecto: Detección de Grietas e Inclinación Structural
# ==============================================================================

# 1. Montar Google Drive
from google.colab import drive
import os
import subprocess
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

drive.mount('/content/drive')

# Rutas de Google Drive y Local
ruta_rar_drive = "/content/drive/MyDrive/Proyecto_Grietas/data/Concrete Crack Images for Classification.rar"
ruta_local_data = "/content/data_local"
ruta_guardado_modelo = "/content/drive/MyDrive/Proyecto_Grietas/Models/modelo_linea_base.keras"

# 2. Descompresión rápida en almacenamiento local de Colab
print("1. Instalando unrar y preparando descompresión local...")
!apt-get -qq update
!apt-get -qq install unrar

os.makedirs(ruta_local_data, exist_ok=True)

print("2. Descomprimiendo dataset localmente para entrenamiento rápido...")
subprocess.run([
    "unrar",
    "x",
    "-o+",
    ruta_rar_drive,
    ruta_local_data
], check=True)

print("Dataset listo en almacenamiento local.")

# 3. Configuración de generadores de datos
IMG_SIZE = (128, 128)
BATCH_SIZE = 64

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

print("\n--- Cargando Datos de Entrenamiento y Validación ---")
train_gen = datagen.flow_from_directory(
    ruta_local_data,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    ruta_local_data,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# 4. Arquitectura de la Línea Base (CNN Sencilla de 3 capas)
model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 5. Entrenamiento del Modelo (3 Épocas para Línea Base de Entrega 1)
print("\n--- Iniciando Entrenamiento ---")
history = model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen
)

# 6. Guardado permanente del Modelo en Drive
os.makedirs("/content/drive/MyDrive/Proyecto_Grietas/Models", exist_ok=True)
model.save(ruta_guardado_modelo)
print(f"\n✅ ¡Éxito! El modelo de línea base ha sido guardado permanentemente en:")
print(ruta_guardado_modelo)

Mounted at /content/drive
1. Instalando unrar y preparando descompresión local...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
2. Descomprimiendo dataset localmente para entrenamiento rápido...
Dataset listo en almacenamiento local.

--- Cargando Datos de Entrenamiento y Validación ---
Found 32000 images belonging to 2 classes.
Found 8000 images belonging to 2 classes.


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 826,529 (3.15 MB)

 Trainable params: 826,529 (3.15 MB)

 Non-trainable params: 0 (0.00 B)


--- Iniciando Entrenamiento ---
Epoch 1/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 576s 1s/step - accuracy: 0.9569 - loss: 0.1195 - val_accuracy: 0.9935 - val_loss: 0.0238
Epoch 2/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 566s 1s/step - accuracy: 0.9868 - loss: 0.0454 - val_accuracy: 0.9935 - val_loss: 0.0185
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 544s 1s/step - accuracy: 0.9902 - loss: 0.0348 - val_accuracy: 0.9973 - val_loss: 0.0106

✅ ¡Éxito! El modelo de línea base ha sido guardado permanentemente en:
/content/drive/MyDrive/Proyecto_Grietas/Models/modelo_linea_base.keras
